# NextLearn - Risk model (caughtUp) on OULAD

Trains a `RandomForestClassifier` on **real** Open University outcomes and exports
`rf-risk.joblib`. Replaces the synthetic-label model.

1. Put the OULAD CSVs in `ml/oulad/raw/` (or upload here in Colab). 2. `Run All`.
3. Copy `rf-risk.joblib` -> `ml/models/`, `oulad_analytics.csv` -> `data/` (last cell).

**Contract:** the features in `FEATURES` order, classes `0/1` (`1 = caughtUp`), a tree model for SHAP.

In [1]:
# Pin scikit-learn to the app's range so the joblib pickle loads in the ML service
# (ml/requirements.txt: scikit-learn>=1.7). If loading later warns about versions,
# match this to `python -c "import sklearn; print(sklearn.__version__)"` in the app env.
!pip install -q "scikit-learn>=1.7,<1.8" pandas numpy joblib



[notice] A new release of pip is available: 25.2 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import os, sys, joblib
import numpy as np, pandas as pd
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn.model_selection import train_test_split, cross_val_score, cross_val_predict, GroupKFold
from sklearn.metrics import roc_auc_score, mean_absolute_error

assert os.path.exists('oulad_features.py'), 'Upload ml/oulad/oulad_features.py next to this notebook'
from oulad_features import (build_dataset, to_training_csv,
                            FEATURES, RISK_LABEL, GRADE_LABEL, KEYS, DEMOGRAPHIC_COLS)

# Auto-locate the OULAD CSVs: ml/oulad/raw/ (Colab upload) or the repo's data/OULAD_DATASET/.
RAW_DIR = next((p for p in ['raw', '../../data/OULAD_DATASET', 'data/OULAD_DATASET']
                if os.path.exists(os.path.join(p, 'studentVle.csv'))), 'raw')
print('using RAW_DIR =', RAW_DIR)
MODELS_DIR = '../models' if os.path.isdir('../models') else '.'
DATA_DIR   = '../../data' if os.path.isdir('../../data') else '.'
SEED, CUTOFF_DAY = 42, 90
RF = dict(n_estimators=100, max_depth=10, random_state=SEED, n_jobs=-1)


using RAW_DIR = ../../data/OULAD_DATASET


In [3]:
# include_demographics=True adds raw demographics for the ABLATION only
# (they never enter the deployed model).
df = build_dataset(raw_dir=RAW_DIR, cutoff_day=CUTOFF_DAY, include_demographics=True)
groups = df['id_student'].values   # for leakage-free, student-grouped CV
print('rows:', len(df), '| students:', df['id_student'].nunique())


rows: 25558 | students: 23397


In [4]:
# --- EDA / validation ---
print('missingness (%):'); print((df[DEMOGRAPHIC_COLS].isna().mean()*100).round(1).to_string())
print('\ncorr of each feature with caughtUp (signs should be pedagogically sensible):')
print(df[FEATURES + [RISK_LABEL]].corr(numeric_only=True)[RISK_LABEL].drop(RISK_LABEL).sort_values().round(3).to_string())
df[FEATURES].describe().T.round(3)


missingness (%):
gender                  0.0
region                  0.0
highest_education       0.0
imd_band                3.8
age_band                0.0
num_of_prev_attempts    0.0
studied_credits         0.0
disability              0.0

corr of each feature with caughtUp (signs should be pedagogically sensible):
gapDepth           -0.464
delayWeeks         -0.424
weakSkillRatio     -0.273
loginFrequency      0.363
recencyRatio        0.416
averageScore        0.447
avgFocusScore         NaN
hasAttentionData      NaN


,count,mean,std,min,25%,50%,75%,max
delayWeeks,25558.0,0.602,1.044,0.0,0.000,0.095,1.000,11.714
averageScore,25558.0,73.587,16.931,0.0,64.000,77.667,86.000,100.000
loginFrequency,25558.0,2.263,1.604,0.0,1.011,1.944,3.267,7.078
gapDepth,25558.0,0.143,0.288,0.0,0.000,0.000,0.200,1.000
recencyRatio,25558.0,0.695,0.373,0.0,0.429,0.893,0.964,1.000
weakSkillRatio,25558.0,0.154,0.295,0.0,0.000,0.000,0.200,1.000
avgFocusScore,25558.0,0.000,0.000,0.0,0.000,0.000,0.000,0.000
hasAttentionData,25558.0,0.000,0.000,0.0,0.000,0.000,0.000,0.000


In [5]:
# --- Train + leakage-free evaluation ---
X, y = df[FEATURES].astype(float).values, df[RISK_LABEL].astype(int).values
print('class balance (caughtUp):', np.bincount(y))
Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.2, random_state=SEED, stratify=y)
clf = RandomForestClassifier(**RF).fit(Xtr, ytr)
print(f'held-out          : acc {clf.score(Xte, yte):.3f} | AUC {roc_auc_score(yte, clf.predict_proba(Xte)[:,1]):.3f}')
# Student-grouped CV: no student appears in both train and test (kills the
# leakage from a student sitting several module-presentations).
cv = cross_val_score(clf, X, y, cv=GroupKFold(5), groups=groups, scoring='roc_auc', n_jobs=-1)
print(f'student-grouped CV: AUC {cv.mean():.3f} +/- {cv.std():.3f}  <- leakage-free')


class balance (caughtUp): [10174 15384]
held-out          : acc 0.782 | AUC 0.842
student-grouped CV: AUC 0.849 +/- 0.003  <- leakage-free


In [6]:
# --- Leave-one-cohort-out: train on some presentations, test on an unseen one ---
cohort = (df['code_module'].astype(str) + '-' + df['code_presentation'].astype(str)).values
aucs = []
for tr, te in GroupKFold(5).split(X, y, cohort):
    m = RandomForestClassifier(**RF).fit(X[tr], y[tr])
    aucs.append(roc_auc_score(y[te], m.predict_proba(X[te])[:,1]))
print(f'leave-cohort-out AUC: {np.mean(aucs):.3f} +/- {np.std(aucs):.3f}')


leave-cohort-out AUC: 0.836 +/- 0.028


In [7]:
# --- Demographic ablation (ANALYSIS ONLY - does not change the deployed model) ---
# Quantifies how much OULAD demographics would add. They are kept OUT of
# deployment: the app can't produce them, and gender/imd/age/disability are
# protected attributes that should not drive a risk score.
cat = pd.get_dummies(df[['gender','region','highest_education','imd_band','age_band','disability']].astype('object'), dummy_na=True)
num = df[['num_of_prev_attempts','studied_credits']].apply(pd.to_numeric, errors='coerce').fillna(0)
X_rich = np.hstack([X, num.values.astype(float), cat.values.astype(float)])
a9 = cross_val_score(RandomForestClassifier(**RF), X, y, cv=GroupKFold(5), groups=groups, scoring='roc_auc', n_jobs=-1).mean()
ar = cross_val_score(RandomForestClassifier(**RF), X_rich, y, cv=GroupKFold(5), groups=groups, scoring='roc_auc', n_jobs=-1).mean()
print(f'{len(FEATURES)} behavioural features        : AUC {a9:.3f}')
print(f'+ {X_rich.shape[1]-len(FEATURES)} demographic columns : AUC {ar:.3f}  (lift {ar-a9:+.3f})')


8 behavioural features        : AUC 0.849
+ 43 demographic columns : AUC 0.850  (lift +0.001)


In [8]:
# --- Refit on all data, export the model + SHAP-background CSV ---
clf.fit(X, y)
assert list(clf.classes_) == [0, 1], f'classes must be [0,1] (1=caughtUp), got {clf.classes_}'
risk_path = os.path.join(MODELS_DIR, 'rf-risk.joblib'); joblib.dump(clf, risk_path)
csv_path = to_training_csv(df, os.path.join(DATA_DIR, 'oulad_analytics.csv'))
print('saved:', risk_path, '|', csv_path)


saved: ../models\rf-risk.joblib | ../../data\oulad_analytics.csv


In [9]:
# --- Smoke test: reload exactly as the app does ---
m = joblib.load(risk_path); row = df[FEATURES].iloc[[0]].astype(float).values
assert row.shape[1] == len(FEATURES)
print('P(caughtUp) row0:', float(m.predict_proba(row)[0][list(m.classes_).index(1)]))
print('OK - copy rf-risk.joblib into ml/models/ and oulad_analytics.csv into data/')


P(caughtUp) row0: 0.7851479724562725
OK - copy rf-risk.joblib into ml/models/ and oulad_analytics.csv into data/
